# 🎯 Phonological Error Detection API
## Complete Implementation in Google Colab with Ngrok

This notebook implements a complete phonological error detection system that:
- Recognizes phonemes from audio using Wav2Vec2
- Converts text to phonemes (G2P)
- Aligns sequences to detect errors
- Identifies phonological processes
- Calculates severity scores
- Exposes API via ngrok

**Setup Instructions:**
1. Enable GPU: Runtime → Change runtime type → GPU (T4)
2. Run all cells in order
3. Get ngrok authtoken from https://dashboard.ngrok.com
4. Paste token in Cell 10
5. Copy the public URL from Cell 11


## Step 1: Install Dependencies


In [ ]:
# Install all required packages
!pip install -q transformers torch torchaudio librosa soundfile
!pip install -q phonemizer espeak-ng
!pip install -q dtaidistance numpy scipy
!pip install -q fastapi uvicorn python-multipart
!pip install -q pyngrok

print("✅ All packages installed!")


## Step 2: Import Libraries


In [ ]:
import torch
import torchaudio
import librosa
import numpy as np
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from phonemizer import phonemize
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from pyngrok import ngrok
import tempfile
import os
from pathlib import Path
import json
import threading
import time

print("✅ Libraries imported!")


## Step 3: Load Wav2Vec2 Model


In [ ]:
# Load Wav2Vec2 model for phoneme recognition
print("Loading Wav2Vec2 model... This may take a minute...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
model.eval()
print("✅ Model loaded successfully!")


## Step 4: Helper Functions

### 4.1 Phoneme Recognition


In [ ]:
def recognize_phonemes(audio_path, sampling_rate=16000):
    """Recognize phonemes from audio file."""
    audio, sr = librosa.load(audio_path, sr=sampling_rate)
    inputs = processor(audio, sampling_rate=sampling_rate, return_tensors="pt")
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    probs = torch.nn.functional.softmax(logits, dim=-1)
    max_probs = torch.max(probs, dim=-1)[0]
    confidence = float(torch.mean(max_probs).item())
    
    predicted_ids = torch.argmax(logits, dim=-1)
    phonemes_str = processor.decode(predicted_ids[0])
    phonemes = [p for p in phonemes_str.split() if p not in ['<pad>', '<s>', '</s>']]
    
    return phonemes, confidence


### 4.2 G2P Conversion


In [ ]:
def text_to_phonemes(text):
    """Convert text to phoneme sequence."""
    try:
        phonemes_str = phonemize(
            text,
            language='en-us',
            backend='espeak',
            strip=True,
            preserve_punctuation=False,
            with_stress=False
        )
        return phonemes_str.split()
    except Exception as e:
        print(f"G2P Error: {e}")
        return []


### 4.3 Sequence Alignment


In [ ]:
from dtaidistance import dtw

def align_sequences(expected, predicted):
    """Align expected and predicted phoneme sequences."""
    all_phonemes = list(set(expected + predicted))
    phoneme_to_idx = {p: i for i, p in enumerate(all_phonemes)}
    
    expected_vec = np.array([phoneme_to_idx[p] for p in expected])
    predicted_vec = np.array([phoneme_to_idx[p] for p in predicted])
    
    distance = dtw.distance(expected_vec, predicted_vec)
    path = dtw.warping_path(expected_vec, predicted_vec)
    
    alignment = []
    for exp_idx, pred_idx in path:
        exp_phoneme = expected[exp_idx] if exp_idx < len(expected) else None
        pred_phoneme = predicted[pred_idx] if pred_idx < len(predicted) else None
        
        if exp_phoneme == pred_phoneme:
            op = "match"
        elif exp_phoneme is None:
            op = "insertion"
        elif pred_phoneme is None:
            op = "deletion"
        else:
            op = "substitution"
        
        alignment.append({
            "expected": exp_phoneme,
            "predicted": pred_phoneme,
            "operation": op
        })
    
    return {
        "alignment": alignment,
        "distance": float(distance),
        "expected_length": len(expected),
        "predicted_length": len(predicted)
    }


### 4.4 Phonological Process Detection


In [ ]:
def detect_phonological_processes(alignment_result):
    """Detect phonological processes from alignment."""
    alignment = alignment_result["alignment"]
    detected_processes = []
    
    # Cluster reduction
    for i, align in enumerate(alignment):
        if align["operation"] == "deletion" and i < 2:
            if i > 0 and alignment[i-1]["operation"] in ["match", "substitution"]:
                detected_processes.append({
                    "process_type": "cluster_reduction",
                    "position": "initial",
                    "affected_phonemes": [align["expected"]],
                    "severity_weight": 0.8
                })
                break
    
    # Final consonant deletion
    if alignment and alignment[-1]["operation"] == "deletion":
        detected_processes.append({
            "process_type": "final_consonant_deletion",
            "position": "final",
            "affected_phonemes": [alignment[-1]["expected"]],
            "severity_weight": 0.6
        })
    
    # Substitutions
    substitutions = [a for a in alignment if a["operation"] == "substitution"]
    if substitutions:
        detected_processes.append({
            "process_type": "substitution",
            "position": "various",
            "affected_phonemes": [s["expected"] for s in substitutions],
            "severity_weight": 0.5
        })
    
    return detected_processes


### 4.5 Severity Scoring


In [ ]:
def calculate_severity(alignment_result, detected_processes, confidence):
    """Calculate severity score (0.0 to 1.0)."""
    error_count = sum(1 for a in alignment_result["alignment"] 
                     if a["operation"] != "match")
    total_phonemes = alignment_result["expected_length"]
    error_rate = error_count / total_phonemes if total_phonemes > 0 else 0
    
    process_weight = 0.0
    if detected_processes:
        process_weight = max(p["severity_weight"] for p in detected_processes)
    
    deviation = alignment_result["distance"] / max(
        alignment_result["expected_length"],
        alignment_result["predicted_length"],
        1
    )
    
    confidence_factor = 1.0 - confidence
    
    severity = (
        0.4 * error_rate +
        0.3 * process_weight +
        0.2 * min(deviation, 1.0) +
        0.1 * confidence_factor
    )
    
    return round(min(1.0, max(0.0, severity)), 2)


## Step 5: Create FastAPI Application


In [ ]:
# Initialize FastAPI Application
app = FastAPI(
    title="Phonological Error Detection API",
    description="AI-based phonological error detection for children's speech",
    version="1.0.0"
)

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Health check endpoints
@app.get("/")
async def root():
    return {
        "message": "Phonological Error Detection API",
        "status": "running",
        "version": "1.0.0"
    }

@app.get("/health")
async def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

# Main detection endpoint
@app.post("/api/v1/detect")
async def detect_phonological_errors(
    audio: UploadFile = File(...),
    target_text: str = Form(...)
):
    """Detect phonological errors in child's speech."""
    try:
        # Save uploaded file temporarily
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp_file:
            content = await audio.read()
            tmp_file.write(content)
            tmp_path = tmp_file.name
        
        try:
            # Step 1: Recognize phonemes
            predicted_phonemes, confidence = recognize_phonemes(tmp_path)
            
            # Step 2: Generate canonical phonemes
            expected_phonemes = text_to_phonemes(target_text)
            
            # Step 3: Align sequences
            alignment_result = align_sequences(expected_phonemes, predicted_phonemes)
            
            # Step 4: Detect phonological processes
            detected_processes = detect_phonological_processes(alignment_result)
            
            # Step 5: Calculate severity
            severity = calculate_severity(
                alignment_result,
                detected_processes,
                confidence
            )
            
            # Step 6: Build response
            primary_process = detected_processes[0] if detected_processes else None
            
            result = {
                "error_category": "phonological" if detected_processes else "none",
                "process_type": primary_process["process_type"] if primary_process else None,
                "expected_text": target_text,
                "predicted_text": " ".join(predicted_phonemes),
                "expected_phonemes": expected_phonemes,
                "predicted_phonemes": predicted_phonemes,
                "alignment": alignment_result["alignment"],
                "pattern_position": primary_process["position"] if primary_process else None,
                "affected_unit": primary_process["affected_phonemes"] if primary_process else [],
                "severity": severity,
                "confidence": round(confidence, 2),
                "detected_processes": detected_processes
            }
            
            return JSONResponse(content=result)
        
        finally:
            # Clean up temporary file
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
    
    except Exception as e:
        print(f"Error: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Processing error: {str(e)}")

print("✅ FastAPI app created!")


In [ ]:
# ⚠️ REPLACE WITH YOUR NGROK AUTHTOKEN
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

# Set ngrok authtoken
ngrok.set_auth_token(NGROK_AUTHTOKEN)
print("✅ Ngrok authtoken set!")


## Step 7: Start Server with Ngrok

**⚠️ Keep this cell running to keep the API active!**

After running this cell, you'll get a public URL. Copy it and use it in Postman or your application.


In [ ]:
# Start FastAPI server in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

# Create ngrok tunnel
public_url = ngrok.connect(8000)

print("=" * 60)
print("🚀 SERVER IS RUNNING!")
print("=" * 60)
print(f"📍 Local URL: http://localhost:8000")
print(f"🌐 Public URL: {public_url.public_url}")
print(f"📋 API Endpoint: {public_url.public_url}/api/v1/detect")
print(f"❤️  Health Check: {public_url.public_url}/health")
print("=" * 60)
print("⚠️  Keep this cell running to keep the API active!")
print("⚠️  Copy the Public URL above to use in Postman or your app!")
print("=" * 60)


## Step 8: Test the API (Optional)

Test the API locally in Colab. First, upload a test audio file using: **Files → Upload to session storage**


In [ ]:
# Test the API locally in Colab
import requests

# Change this to your test audio file path (after uploading)
test_audio_path = "/content/your_test_audio.wav"
target_text = "school"

# Prepare request
files = {'audio': open(test_audio_path, 'rb')}
data = {'target_text': target_text}

# Send request
response = requests.post(
    "http://localhost:8000/api/v1/detect",
    files=files,
    data=data
)

print("Response Status:", response.status_code)
print("\nResponse JSON:")
print(json.dumps(response.json(), indent=2))
